# 1. Install Dependencies

In [ ]:
!pip install -U pip
!pip install TTS

# 2. Clone Repository & Setup

In [ ]:
!git clone https://VanModers:@github.com/VanModers/oostfraeisk_text_to_speech

In [ ]:
%cd oostfraeisk_text_to_speech

In [ ]:
!git pull

# 3. A100 GPU Optimizations

In [ ]:
import torch

# Enable TF32 for faster matmul on A100
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark = True

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"TF32 enabled: {torch.backends.cuda.matmul.allow_tf32}")

# 4. Download Pretrained German VITS Model

In [ ]:
from TTS.utils.manage import ModelManager

manager = ModelManager()
model_path, config_path, _ = manager.download_model("tts_models/de/thorsten/vits")
print(f"Model: {model_path}")
print(f"Config: {config_path}")

# 5. Define East Frisian Character Set

For grapheme-only training, we need to define all characters that appear in the dataset.

In [ ]:
from TTS.tts.configs.vits_config import VitsConfig
from TTS.tts.configs.shared_configs import CharactersConfig

# All characters that appear in East Frisian text
# Standard letters
letters_lower = "abcdefghijklmnopqrstuvwxyz"
letters_upper = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

# East Frisian special characters (including circumflex vowels)
# â ê î ô û = extra-long vowels
# ä ö ü = umlauts
# ó = open o sound
# ğ = velar fricative (like Dutch g)
east_frisian_special = "ÂÊÎÔÛâêîôûÄÖÜäöüÓóĞğß"

# Numbers (in case they appear in text)
numbers = "0123456789"

# All grapheme characters (no punctuation here)
all_characters = letters_lower + letters_upper + east_frisian_special + numbers

# Punctuation marks
punctuations = "!\"'(),-.:;?  \n"

print(f"Character set ({len(all_characters)} chars): {all_characters}")
print(f"Punctuations: {repr(punctuations)}")

# 6. Configure Model for Grapheme-Only Training

In [ ]:
# Create custom character config for East Frisian graphemes
characters_config = CharactersConfig(
    characters_class="TTS.tts.utils.text.characters.Graphemes",
    characters=all_characters,
    punctuations=punctuations,
    pad="<PAD>",
    eos="<EOS>",
    bos="<BOS>",
    blank="<BLNK>",
)

# Load pretrained config as base
config = VitsConfig()
config.load_json(config_path)

# === CRITICAL: Use graphemes, not phonemes ===
config.use_phonemes = False
config.characters = characters_config

# Dataset settings
config.datasets[0].formatter = "ljspeech"
config.datasets[0].path = "data/oostfraeisk"
config.datasets[0].meta_file_train = "metadata.csv"

# Training settings
config.output_path = "tts_train_dir_grapheme"
config.epochs = 100
config.run_eval = True
config.test_delay_epochs = -1

# === A100 OPTIMIZED SETTINGS ===
config.batch_size = 8 
config.eval_batch_size = 16
config.num_loader_workers = 8 
config.num_eval_loader_workers = 4

config.save_step = 500

# Logging
config.print_step = 100
config.log_model_step = 1000

# Mixed precision & cuDNN
config.mixed_precision = True
config.cudnn_benchmark = True

print("=" * 50)
print("GRAPHEME-ONLY TRAINING CONFIG")
print("=" * 50)
print(f"use_phonemes: {config.use_phonemes}")
print(f"characters_class: {config.characters.characters_class}")
print(f"Batch size: {config.batch_size}")
print(f"Learning rate: {config.lr_gen}")
print(f"Output path: {config.output_path}")

# 7. Verify Dataset

Check that all characters in the dataset are covered by our character set.

In [ ]:
from pathlib import Path

metadata_path = Path("data/oostfraeisk/metadata.csv")

# Read all text from metadata
all_text = ""
with open(metadata_path, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.strip().split('|')
        if len(parts) >= 2:
            all_text += parts[1] + " "

# Find unique characters in dataset
dataset_chars = set(all_text)
config_chars = set(all_characters + punctuations)

# Check for missing characters
missing = dataset_chars - config_chars
if missing:
    print(f"⚠️  Missing characters in config: {missing}")
    print(f"   Add these to 'all_characters' or 'punctuations'")
else:
    print(f"✓ All {len(dataset_chars)} unique characters are covered!")

# Show character frequency
from collections import Counter
char_freq = Counter(all_text)
print(f"\nMost common characters:")
for char, count in char_freq.most_common(20):
    print(f"  '{char}': {count}")

# 8. Initialize Model and Tokenizer

In [ ]:
from TTS.tts.models.vits import Vits
from TTS.utils.audio import AudioProcessor
from TTS.tts.utils.text.tokenizer import TTSTokenizer

# Initialize audio processor and tokenizer
ap = AudioProcessor.init_from_config(config)
tokenizer, config = TTSTokenizer.init_from_config(config)

print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Vocab size: {tokenizer.characters.num_chars}")
print(f"Characters: {tokenizer.characters.characters}")

In [ ]:
# Test tokenization with East Frisian text
test_sentences = [
    "Moin, woo gaajt 't dii?",
    "Däi süen skint up us land.",
    "Wii prootent Oostfräisk.",
    "Hest duu däi süen fandóóeğ al säin?",
]

print("Tokenization test:")
print("=" * 50)
for sent in test_sentences:
    try:
        ids = tokenizer.text_to_ids(sent)
        back = tokenizer.ids_to_text(ids)
        print(f"Input:  {sent}")
        print(f"IDs:    {ids[:20]}..." if len(ids) > 20 else f"IDs:    {ids}")
        print(f"Back:   {back}")
        print()
    except Exception as e:
        print(f"ERROR with '{sent}': {e}")
        print()

# 9. Load Pretrained Model

We load the German VITS model but with `strict=False` since the embedding layer size will differ (graphemes vs phonemes).

In [ ]:
# Initialize model with our grapheme config
model = Vits.init_from_config(config)

# Load pretrained weights (strict=False allows different embedding sizes)
model.load_checkpoint(config, model_path, eval=False, strict=False)

print(f"Model loaded successfully!")
print(f"Embedding input dim: {model.text_encoder.emb.num_embeddings}")

# 10. Train the Model

In [ ]:
from TTS.tts.datasets import load_tts_samples
from trainer import Trainer, TrainerArgs

# Load dataset
train_samples, eval_samples = load_tts_samples(
    config.datasets[0],
    eval_split=True,
    eval_split_max_size=config.eval_split_max_size,
    eval_split_size=config.eval_split_size,
)

print(f"Train samples: {len(train_samples)}")
print(f"Eval samples: {len(eval_samples)}")

# Initialize trainer
trainer = Trainer(
    TrainerArgs(),
    config,
    config.output_path,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

# Start training
trainer.fit()

# 11. Find Best Checkpoint

In [ ]:
import glob

output_path = "tts_train_dir_grapheme"
ckpts = sorted(glob.glob(output_path + "/*/*.pth"))
configs = sorted(glob.glob(output_path + "/*/*.json"))

print("Available checkpoints:")
for ckpt in ckpts[-5:]:
    print(f"  {ckpt}")

print(f"\nLatest config: {configs[-1] if configs else 'None'}")

# 12. Test Inference

In [ ]:
# Update these paths to your best checkpoint
best_checkpoint = ckpts[-1] if ckpts else None
best_config = configs[-1] if configs else None

if best_checkpoint:
    print(f"Using checkpoint: {best_checkpoint}")
    print(f"Using config: {best_config}")

In [ ]:
# Generate speech
!tts --text "Moin, woo góót 't dii? Ik hoop, dat 't dii gaud góót." \
     --model_path {best_checkpoint} \
     --config_path {best_config} \
     --out_path out_grapheme.wav

In [ ]:
import IPython
IPython.display.Audio("out_grapheme.wav")

# 13. Copy Best Model to model/ Directory

In [ ]:
!mkdir -p model_grapheme

In [ ]:
# Copy your best checkpoint (update the path as needed)
# Example: !cp tts_train_dir_grapheme/vits-.../checkpoint_10000.pth model_grapheme/model.pth
!cp {best_checkpoint} model_grapheme/model.pth
!cp {best_config} model_grapheme/config.json

# 14. Push to GitHub

In [ ]:
!git lfs install
!git lfs track "*.pth"

In [ ]:
!git config --global user.email "programmingstudios227@gmail.com"
!git config --global user.name "Tido Specht"
!git add -A
!git commit -m "Grapheme-only model treenäärt"

In [ ]:
!git push

# 15. Push to HuggingFace (Optional)

In [ ]:
!pip install huggingface_hub

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Clone your HuggingFace space
%cd ..
!git clone https://huggingface.co/spaces/VanModers114/East_Frisian_TTS

In [ ]:
# Copy model to HuggingFace repo
!cp oostfraeisk_text_to_speech/model_grapheme/model.pth East_Frisian_TTS/model.pth
!cp oostfraeisk_text_to_speech/model_grapheme/config.json East_Frisian_TTS/config.json

In [ ]:
%cd East_Frisian_TTS
!git lfs install
!git lfs track "*.pth"
!git add -A
!git commit -m "Grapheme-only model"
!git push